# Training: five-target GIN regression on commuting graphs

The data are loaded with a fixed seed and split stratified by group order, and each target is standardized separately.

In [ ]:
import sys
from pathlib import Path
# locate src/ whether the kernel starts in notebooks/ or the repo root
_root = Path.cwd().resolve()
_src = _root.parent / 'src' if (_root.parent / 'src').is_dir() else _root / 'src'
sys.path.insert(0, str(_src))
_repo = _src.parent
DATA_PATH = _repo / 'data' / 'commuting_pgroups_p2.json'
CHECKPOINT_PATH = _repo / 'notebooks' / 'best_model.pt'

import torch
from torch_geometric.loader import DataLoader
from dataset import PGroupDataset
from model import PGroupGNN
from sklearn.preprocessing import StandardScaler
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

use_ph = True
batch_size = 16
epochs = 200
patience = 20

dataset = PGroupDataset(str(DATA_PATH), use_ph=use_ph, seed=SEED, stratify_by_order=True)
dataset.report_split_by_order()  # prints train/val/test counts per order (orders with fewer than 10 groups are all in training)

train_ys = torch.stack([dataset[i].y for i in dataset.train_idx]).view(-1, 5)
scalers = [StandardScaler().fit(train_ys[:, j].reshape(-1, 1)) for j in range(5)]

def normalize(y_batch):
    y_batch = y_batch.view(-1, 5).cpu().numpy()
    y_scaled = np.hstack([scalers[j].transform(y_batch[:, [j]]) for j in range(5)])
    return torch.tensor(y_scaled, dtype=torch.float)

def denormalize(pred_batch):
    pred_batch = pred_batch.view(-1, 5).cpu().numpy()
    y_inv = np.hstack([scalers[j].inverse_transform(pred_batch[:, [j]]) for j in range(5)])
    return torch.tensor(y_inv, dtype=torch.float)

train_loader = DataLoader([dataset[i] for i in dataset.train_idx], batch_size=batch_size, shuffle=True)
val_loader   = DataLoader([dataset[i] for i in dataset.val_idx],   batch_size=batch_size)
test_loader  = DataLoader([dataset[i] for i in dataset.test_idx],  batch_size=batch_size)

## Model, optimizer, loss

In [ ]:
model = PGroupGNN(node_dim=dataset[0].x.shape[1], ph_dim=100 if use_ph else 0, out_dim=5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.HuberLoss()

## Training loop
Training stops once validation loss has not improved for 20 epochs. The model with the lowest validation loss is saved.

In [ ]:
best_val_loss = float('inf')
counter = 0

print(f"Training on {len(train_loader.dataset)} graphs, validating on {len(val_loader.dataset)}")

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        y_norm = normalize(batch.y)
        pred = model(batch.x, batch.edge_index, batch.batch, batch.ph_features)
        loss = criterion(pred, y_norm)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            y_norm = normalize(batch.y)
            pred = model(batch.x, batch.edge_index, batch.batch, batch.ph_features)
            val_loss += criterion(pred, y_norm).item()
    val_loss /= len(val_loader)
    
    print(f"Epoch {epoch:3d} | Train loss: {train_loss/len(train_loader):.4f} | Val loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save({"state_dict": model.state_dict(),
                    "config": {"node_dim": dataset[0].x.shape[1],
                               "ph_dim": 100 if use_ph else 0,
                               "out_dim": 5, "hidden_dim": 128, "pooling": "mean"},
                    "log_size_targets": True}, CHECKPOINT_PATH)
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

## Test evaluation

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH)
model = PGroupGNN(**checkpoint["config"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()
test_preds, test_trues = [], []
with torch.no_grad():
    for batch in test_loader:
        pred = denormalize(model(batch.x, batch.edge_index, batch.batch, batch.ph_features))
        test_preds.append(pred)
        test_trues.append(batch.y.view(-1, 5))

test_preds = torch.cat(test_preds).numpy()
test_trues = torch.cat(test_trues).numpy()

mses = [np.mean((test_preds[:, i] - test_trues[:, i])**2) for i in range(5)]

## Closed-form baselines
Centre size and the number of conjugacy classes k(G) can be computed exactly from the node features. The MSE of these exact values is printed next to the model's MSE as a baseline.

In [ ]:
# Baselines on the test set, computed from the node features:
# centre size = sum(is_central) per graph, k(G) = mean(centralizer_size) per graph.
from torch_geometric.nn import global_add_pool, global_mean_pool

baseline_centre, baseline_kG, true_centre, true_kG = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        is_central_col = batch.x[:, 3:4]
        centralizer_col = batch.x[:, 1:2]
        raw_centre = global_add_pool(is_central_col, batch.batch).view(-1)
        # the centre size target is stored as log2 of the count (log_size_targets=True, p=2)
        baseline_centre.append(torch.log2(raw_centre))
        baseline_kG.append(global_mean_pool(centralizer_col, batch.batch).view(-1))
        y = batch.y.view(-1, 5)
        true_centre.append(y[:, 2])
        true_kG.append(y[:, 4])

baseline_centre = torch.cat(baseline_centre).numpy()
baseline_kG = torch.cat(baseline_kG).numpy()
true_centre = torch.cat(true_centre).numpy()
true_kG = torch.cat(true_kG).numpy()

mse_baseline_centre = np.mean((baseline_centre - true_centre) ** 2)
mse_baseline_kG = np.mean((baseline_kG - true_kG) ** 2)

print("\nTest results")
print(f"Nilpotency class MSE:  {mses[0]:.3f}")
print(f"Derived length MSE:    {mses[1]:.3f}")
print(f"Centre size MSE (log2 units): {mses[2]:.3f}   (baseline: {mse_baseline_centre:.3f})")
print(f"Commutator size MSE (log2 units): {mses[3]:.3f}")
print(f"Conjugacy classes MSE: {mses[4]:.3f}   (baseline: {mse_baseline_kG:.3f})")